```bash
source ~/.venv-vllm-metal/bin/activate
vllm serve Qwen/Qwen3-0.6B \
  --host 127.0.0.1 \
  --port 8000 \
  --api-key local-key \
  --default-chat-template-kwargs '{"enable_thinking": false}'
```

In [6]:
import subprocess
import time
import requests
import pandas as pd
from openai import OpenAI

MODEL = "Qwen/Qwen3-0.6B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"
API_KEY = "local-key"


# def wait_for_server(timeout=300):
#     start = time.time()

#     while time.time() - start < timeout:
#         try:
#             r = requests.get(f"{BASE_URL}/models", timeout=5)
#             if r.status_code == 200:
#                 print("vLLM server is ready.")
#                 return
#         except requests.exceptions.RequestException:
#             pass

#         time.sleep(5)

#     raise TimeoutError("vLLM server did not start in time.")


def get_response(client, prompt):
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=512,
    )

    return completion.choices[0].message.content


def run_batch(df):
    client = OpenAI(
        base_url=BASE_URL,
        api_key=API_KEY,
    )

    rows = []

    for prompt in df["prompt"]:
        try:
            response = get_response(client, prompt)
        except Exception as e:
            response = f"ERROR: {e}"

        rows.append({
            "prompt": prompt,
            "response": response,
        })

    return pd.DataFrame(rows)

In [ ]:
# df = pd.DataFrame({
#         "prompt": [
#             "Explain logistic regression in one paragraph.",
#             "What is the difference between LoRA and QLoRA?",
#         ]
#     })

df = pd.read_parquet("hf://datasets/LLM-LAT/harmful-dataset/data/train-00000-of-00001.parquet")

try:
    results_df = run_batch(df)

    results_df.to_csv("../data/qwen3_responses.csv", index=True)
    print(results_df)
finally:
    print("Done")   